# Optimize performance in dataloading

In [1]:
import torch
from typing import Any, Dict, Optional, Tuple

from cv2 import transform
import torch
from lightning import LightningDataModule
from torch.utils.data import ConcatDataset, DataLoader, Dataset, random_split
from torchvision.transforms import transforms
import tonic
import numpy as np
import sys
sys.path.append("../src/utils")
sys.path.append("../src/data/")
sys.path.append("../src/data/components/")
sys.path.append("../src/models/components/")
import eventIO, event_represenations
from TOPSPIN import Hdf5Dataset
import fire_net
from topspin_datamodule import TopspinDataModule

import time
import pandas as pd

/home/lkolmar/anaconda3/envs/learning/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
data_module = TopspinDataModule(
    data_dir="/data/lkolmar/datasets/topspin_fit_to_max/",
    time_window=5000,  # in us
    num_bins=10,
    sensor_size=(100, 100),
    train_val_test_split=(1294, 277, 277),  # n = 1848 (70%, 15%, 15%)
    batch_size=8,
    num_workers=1,
    pin_memory=True,
)
data_module.prepare_data()
data_module.setup()
dataloader = data_module.train_dataloader()

In [3]:
durations = []
n = 10
for i in range(n):
    start = time.time()
    batch = dataloader.__iter__().__next__()
    end = time.time()
    durations.append(end - start)
    print(f"Batch {i+1}/{n} took {end - start:.4f} seconds")
    #time.sleep(6)  # Sleep to simulate processing time

avg_duration = np.mean(durations)
print(f"Average duration for loading a batch: {avg_duration:.4f} seconds")
print(f"Total time for {n} batches: {np.sum(durations):.4f} seconds")

Batch 1/10 took 2.5069 seconds
Batch 2/10 took 2.6026 seconds
Batch 3/10 took 2.3874 seconds
Batch 4/10 took 2.6075 seconds
Batch 5/10 took 2.4496 seconds
Batch 6/10 took 2.5778 seconds
Batch 7/10 took 2.4297 seconds
Batch 8/10 took 2.5897 seconds
Batch 9/10 took 2.4244 seconds
Batch 10/10 took 2.5514 seconds
Average duration for loading a batch: 2.5127 seconds
Total time for 10 batches: 25.1270 seconds


## Average duration per batch: 4s
reducing num_workers helps and gives speedup up to 2s

## Benchmark the getitem function

In [4]:
events_struct = np.dtype(
    [("x", np.int16), ("y", np.int16), ("t", np.int64), ("p", np.int8)]
)

In [5]:
labels = pd.read_csv("/data/lkolmar/datasets/topspin_fit_to_max/config/labels.csv")
transform = transforms.Compose([
    lambda ev: event_represenations.create_sequence(ev,
                                                    time_window=5000,
                                                    num_bins=10,
                                                    sensor_size=(100, 100),
    )
])

In [6]:
def get_item(idx):
    index_str = str(idx).zfill(5)
    start = time.time()
    events = eventIO.load_hdf5(f"/data/lkolmar/datasets/topspin_fit_to_max/preprocessed/{index_str}/{index_str}_roi.hdf5")
    end = time.time()
    print(f"Loading events took {end - start:.4f} seconds for index {idx}")
    start = time.time()
    array = np.empty_like(events.get_x(), dtype=events_struct)
    array["x"] = events.get_x()
    array["y"] = events.get_y()
    array["t"] = events.get_ts()
    array["p"] = events.get_p()
    end = time.time()
    print(f"Creating array took {end - start:.4f} seconds for index {idx}")
    start = time.time()
    
    array = transform(array)
    label = labels.loc[labels['index'] == idx, 'label'].values[0]

    end = time.time()
    print(f"Transforming array took {end - start:.4f} seconds for index {idx}")
    return array, label

n = 10
durations = []
for i in range(n):
    start = time.time()
    item = get_item(i)
    end = time.time()
    durations.append(end - start)
    # print(f"Item {i+1}/{n} took {end - start:.4f} seconds")

avg_duration = np.mean(durations)
print(f"Average duration for get_item: {avg_duration:.4f} seconds")
print(f"Total time for {n} items: {np.sum(durations):.4f} seconds")

Loading events took 0.1069 seconds for index 0
Creating array took 0.0042 seconds for index 0
Transforming array took 0.0828 seconds for index 0
Loading events took 0.0834 seconds for index 1
Creating array took 0.0036 seconds for index 1
Transforming array took 0.0800 seconds for index 1
Loading events took 0.0866 seconds for index 2
Creating array took 0.0033 seconds for index 2
Transforming array took 0.0665 seconds for index 2
Loading events took 0.0903 seconds for index 3
Creating array took 0.0037 seconds for index 3
Transforming array took 0.0736 seconds for index 3
Loading events took 0.0861 seconds for index 4
Creating array took 0.0033 seconds for index 4
Transforming array took 0.0451 seconds for index 4
Loading events took 0.0871 seconds for index 5
Creating array took 0.0038 seconds for index 5
Transforming array took 0.0802 seconds for index 5
Loading events took 0.0858 seconds for index 6
Creating array took 0.0031 seconds for index 6
Transforming array took 0.0511 secon

Loading the hdf5 takes half the time <br>
transofrming it takes the other half

In [9]:
import subcomponents
firenet = fire_net.FireNet(
    input_channels=10,
    hidden_channels=16,
    kernel_size=3,
    head=subcomponents.ClassificationHead(16, (100,100),num_classes=6))

start = time.time()
batch = next(iter(dataloader))
end = time.time()
print(f"Loading batch took {end - start:.4f} seconds") 

start = time.time()
output = firenet(batch[0])
end = time.time()
print(f"Forward pass took {end - start:.4f} seconds")

Loading batch took 2.7023 seconds
Forward pass took 1.8394 seconds
